In [3]:
import sys
!{sys.executable} -m pip install matplotlib


  Using cached matplotlib-3.9.4-cp39-cp39-win_amd64.whl (7.8 MB)
  Using cached kiwisolver-1.4.7-cp39-cp39-win_amd64.whl (55 kB)
  Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
     ---------------------------------------- 2.2/2.2 MB 4.5 MB/s eta 0:00:00
  Using cached contourpy-1.3.0-cp39-cp39-win_amd64.whl (211 kB)
     ---------------------------------------- 2.7/2.7 MB 2.5 MB/s eta 0:00:00
  Using cached importlib_resources-6.5.2-py3-none-any.whl (37 kB)
     ------------------------------------ 111.1/111.1 KB 646.7 kB/s eta 0:00:00


You should consider upgrading via the 'D:\AB\Barclay\mlmodel\spiketest\Scripts\python.exe -m pip install --upgrade pip' command.


In [7]:
# -------------- Import Libraries --------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
import os

# -------------- Load Data --------------
df = pd.read_csv('data/train_logs.csv')

# -------------- Prepare Data --------------
# Convert Timestamp to datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values('Timestamp')

# Only focus on Response_Time for now
response_times = df['Response_Time'].values.reshape(-1, 1)

# Scale data between 0 and 1
scaler = MinMaxScaler()
response_times_scaled = scaler.fit_transform(response_times)

# -------------- Create Sequences --------------
def create_sequences(data, time_steps=30):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:i+time_steps])
        y.append(data[i+time_steps])
    return np.array(X), np.array(y)

time_steps = 30
X, y = create_sequences(response_times_scaled, time_steps)

# -------------- Build LSTM Model --------------
model = Sequential([
    LSTM(64, activation='relu', input_shape=(X.shape[1], X.shape[2]), return_sequences=True),
    Dropout(0.2),
    LSTM(32, activation='relu', return_sequences=False),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

# -------------- Train Model --------------
history = model.fit(X, y, epochs=50, batch_size=32, validation_split=0.2, shuffle=False)




Epoch 1/50


D:\AB\Barclay\mlmodel\spiketest\lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 0.1375 - val_loss: 5.3830e-04
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0991 - val_loss: 9.4145e-04
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0689 - val_loss: 5.2267e-04
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0459 - val_loss: 5.2304e-04
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.1125 - val_loss: 0.0011
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.1002 - val_loss: 5.2100e-04
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0839 - val_loss: 6.3444e-04
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0641 - val_loss: 9.8222e-04
Epoch 9/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0437 - val_loss: 7.9604e-04
Epoch 10/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0207 - val_loss: 5.3588e-04
Epoch 11/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0214 - val_loss: 5.5626e-04
Epoch 12/50
25/25 ━━━━━━━━━━━━━━━━

In [8]:
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mse'])
model.save('lstm_anomaly_model.h5')
print("Model saved as lstm_anomaly_model.h5")

# Save the scaler too
import joblib
joblib.dump(scaler, 'scaler.save')

Model saved as lstm_anomaly_model.h5


['scaler.save']